# Introduction
GOAL: To train a model to predict the wild fire risk of properties, using census data as input features and the proximity to fires as a target feature.

It would be cool to add in data regarding local climate.

In [1]:
import census
from census import Census

import censusgeocode as cg
# import pytidycensus as tc

import geopandas as gpd
import json
import os
import numpy as np
import pandas as pd
import seaborn as sns
from datetime import datetime
import sqlalchemy as sql
# import matplotlib.pyplot as plt
# import pandas as pd
# from censusdis.states import ALL_STATES_AND_DC

import load_wildfires
import load_census
import load_properties

from censusgeocode import CensusGeocode
from random_address import real_random_address
from sqlalchemy.engine import URL

from IPython.display import IFrame
# from pygris import tracts
from matplotlib.colors import to_hex
from scipy.stats import randint, uniform
from pathlib import Path
# from shapely.geometry import Point
from shapely.ops import nearest_points
from shapely import distance
# from tqdm.notebook import tqdm
import gis

from sklearn.ensemble import RandomForestRegressor 
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import make_scorer, mean_poisson_deviance, mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PowerTransformer
from xgboost import XGBRegressor
from settings import GIS_DESIRED_COLS, GIS_DEFAULT_CRS, SQL_ENGINE_STR
from sql_funcs import SQL

from sqlalchemy import text





In [2]:
sql_obj = SQL(test=True)
#TODO: Remove this when you're done testing
# try:
#     sql_obj._drop_table('census_cache', True)
# except RuntimeError as e:
#     print("census_cache table did not exist.")
#     pass

# Load Properties

## Select Properties of Interest

Chosing 300000 properties randomly from US addresses. We will join relevant census data to these addresses. This will probably take awhile, so best to run it overnight.

We don't care about the address itself. We add a census identifier called the GEOID which based on the coordinate's state, county, and tract number.

Using a package that makes use of the [US Census Geocoder API](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/census-geocoder.html), requests can be in batches of 10,000.

https://pypi.org/project/random-address/

TODO: This feels like a very naive approach, atm. Takes about 2s per address. Yuck. Speed it up.

In [3]:
props = load_properties.Properties(sql_obj=sql_obj)
# props.add_random_properties(100)


'properties_test' table found.


In [4]:
properties = props.get_properties_gpd()

properties.tail()

,geoid,block_id,block_grp,tract_id,county_id,state_id,geometry
114,250214008004001,4001,4,400800,21,25,POINT (-71.123 42.338)
115,120050015012006,2006,2,1501,5,12,POINT (-85.669 30.219)
116,130510119001098,1098,1,11900,51,13,POINT (-81.092 32.065)
117,510131011004015,4015,4,101100,13,51,POINT (-77.155 38.881)
118,130510110054001,4001,4,11005,51,13,POINT (-81.062 31.938)


# Load Features from US Census

2023 US Census Data

Using an API key, we will use the 'census' Python package to interact with the US Govermnent's census API.

In [5]:
census = load_census.CensusData(sql_obj=sql_obj, year=2023, granularity='county')


In [6]:
combined_gdf = census.merge_census_info(properties)
combined_gdf.head()


23 geographies needed, 0 cached, 23 to fetch.
Unique geoids: 23.
[0] Cache hit
[1] Cache hit
[2] Cache hit
[3] Cache hit
[4] Cache hit
[5] Cache hit
[7] Cache hit
[8] Cache hit
[10] Cache hit
[14] Cache hit
[17] Cache hit
[20] Cache hit
[22] Cache hit
[23] Cache hit
[24] Cache hit
[35] Querying id 09003 at county...
[36] Cache hit
[38] Cache hit
[41] Cache hit
[42] Cache hit
[44] Cache hit
[45] Cache hit
[47] Cache hit


,geoid,geometry,B25014H_002E,B25014H_003E,B25014H_001E,B25014C_001E,B25014C_002E,B25014C_003E,B25014_002E,B25014_003E,...,B08203_027E,B08203_029E,B08203_030E,B25031_002E,B25031_001E,B25031_006E,B25031_005E,B25031_004E,B25031_003E,B25031_007E
0,240037027021018,POINT (-76.531 39.022),0.104455,0.000756,0.105210,0.000535,0.000524,0.000010,0.115166,0.094083,...,728.0,7613.0,8118.0,0.118794,0.132437,0.170904,0.149807,0.129309,0.109277,0.189472
1,40130929002013,POINT (-112.186 33.529),0.092410,0.001398,0.093808,0.002212,0.001915,0.000297,0.096752,0.071906,...,7152.0,55092.0,54246.0,0.101749,0.129077,0.172428,0.152501,0.127776,0.109394,0.207076
2,211110112023003,POINT (-85.675 38.198),0.104295,0.000897,0.105191,0.000168,0.000164,0.000004,0.095415,0.080534,...,1421.0,7349.0,4988.0,0.103243,0.130331,0.179042,0.158370,0.133064,0.108709,0.187240
3,11010029012000,POINT (-86.243 32.315),0.059497,0.000466,0.059963,0.000285,0.000285,0.000000,0.100376,0.083402,...,288.0,1230.0,1775.0,0.113133,0.138929,0.177033,0.153725,0.128585,0.109336,0.179259
4,110010019014008,POINT (-77.024 38.968),0.067891,0.001048,0.068940,0.000621,0.000613,0.000007,0.068471,0.055491,...,3508.0,2078.0,1178.0,0.116624,0.125686,0.195872,0.148773,0.128928,0.124694,0.159423


# Load Wildfire GIS Data for 2024

We will use point data from the Visible Infrared Imaging Radiometer Suite (VIIRS). A valid alternative is using burn boundary data. There are a few different data sources we could use, but in the interest of (portfolio) simplicity we'll use just the VIIRS.

N:B: May be a good chance to practice using AWS DB storage and retrieval?

In [7]:
wildfires = load_wildfires.WildfireData(sql_obj=sql_obj)


Extracting wildfire data from GIS files.
Loading satellite:  J1V-C2
Loading satellite:  J2V-C2
Loading satellite:  LS
Loading satellite:  M-C61
Loading satellite:  SV-C2


## Create Target (Wildfire Proximity Score)

Give Each Property a Wildfire Risk Score based on the proximity to wildfires.



In [8]:
proximity_features = gis.calc_all_features(combined_gdf, wildfires.data)
targets_features = pd.concat([combined_gdf, proximity_features], axis=1)

In [9]:
targets_features.head()

,geoid,geometry,B25014H_002E,B25014H_003E,B25014H_001E,B25014C_001E,B25014C_002E,B25014C_003E,B25014_002E,B25014_003E,...,exp_decay_score,fire_count_0_10km,fire_count_10_25km,fire_count_25_50km,fire_count_50_100km,fire_FRP_0_10km,fire_FRP_10_25km,fire_FRP_25_50km,fire_FRP_50_100km,nearest_fire_km
0,240037027021018,POINT (-76.531 39.022),0.104455,0.000756,0.105210,0.000535,0.000524,0.000010,0.115166,0.094083,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,360.247081
1,40130929002013,POINT (-112.186 33.529),0.092410,0.001398,0.093808,0.002212,0.001915,0.000297,0.096752,0.071906,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,360.238669
2,211110112023003,POINT (-85.675 38.198),0.104295,0.000897,0.105191,0.000168,0.000164,0.000004,0.095415,0.080534,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,360.244382
3,11010029012000,POINT (-86.243 32.315),0.059497,0.000466,0.059963,0.000285,0.000285,0.000000,0.100376,0.083402,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,360.249612
4,110010019014008,POINT (-77.024 38.968),0.067891,0.001048,0.068940,0.000621,0.000613,0.000007,0.068471,0.055491,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,360.246944


# Machine Learning Considerations
## Scoring Methods

For the float risk score, we can use Mean Squared Error (MSE) or Root Mean Squared Error (RMSE). Since it's quadratic in difference between observations and predictions deviations, MSE strongly penalizes large misses, which would be expensive for the insurance company.

For the risk category counts, they appear to be Poisson distributed, so a Poisson loss-function is appropriate.

For any classification model with the binned risk categories, we want to make large misses costly (i.e. predicting a 1 when the category is a 10), since these would also be very costly to the insurance company. To be honest, MSE will work here as well, since the categories are just 

# Model Machine Learning


NB: A good chance to make use of AWS compute.


### Split Data into Features/Targets



In [10]:
# Target columns
target_col = ml_df['sat_avg']
feature_cols = ml_df.drop(columns=['sat_avg'])

random_state = 77
X_train, X_test, y_train, y_test = train_test_split(feature_cols, target_col, test_size=0.2, random_state=random_state)


NameError: name 'ml_df' is not defined

#### RandomForestRegressor


In [ ]:
# rfr_rand_fp =os.path.join("Models","model_pred_best_RFR_randomCV.sav")

# if not os.path.exists(rfr_rand_fp):
#     param_dist = {
#         "n_estimators":    randint(100, 1000),   
#         "max_depth":       randint(5, 50),       
#         "min_samples_split": randint(2, 11),    
#         "min_samples_leaf":  randint(1, 5), 
#         "max_features":    [ "sqrt", "log2"] 
#     }
    
#     rfr = RandomForestRegressor(random_state=random_state, n_jobs=-1)
    
#     random_srch = RandomizedSearchCV(
#         estimator=rfr,
#         param_distributions=param_dist,
#         n_iter=5,  # start with 20 to get a feel for time
#         scoring='neg_mean_squared_error',
#         cv=5, 
#         random_state=random_state,
#         n_jobs=-1,
#         verbose=5  # 1
#     )

#     random_srch.fit(X_train, y_train)
    
#     print('best rfr params:', random_srch.best_params_)
#     # print('best score:', -random_srch.best_score_)
#     best_rfr = random_srch.best_estimator_
    

#     pickle.dump(best_rfr, open(rfr_rand_fp, 'wb'))
# best_rfr = pickle.load(open(rfr_rand_fp,'rb'))

# y_pred = best_rfr.predict(X_train)
# print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred)))

# y_pred = best_rfr.predict(X_test)
# print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


#### XGBoost


In [ ]:
cv_choice = 3
n_job_choice = 20

xgb_rand_fp =os.path.join("Models","model_pred_bbest_XGB_randomCV_{}cv_{}job.sav".format(cv_choice, n_job_choice))

if not os.path.exists(xgb_rand_fp):
    
    # XGBRegressor
    param_dist = {
        'n_estimators':randint(100, 1000),
        'learning_rate':uniform(0.01, 0.29),
        'max_depth':randint(3, 12),
        'min_child_weight':randint(1, 10),
        'subsample':uniform(0.5, 0.5),
        'colsample_bytree':uniform(0.5, 0.5),
        'gamma':uniform(0, 0.5),
        'reg_alpha': uniform(0, 1),
        'reg_lambda':uniform(0, 1),
    }
    
    xgb = XGBRegressor(random_state=random_state, n_jobs=-1)
    
    random_srch = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_dist,
        n_iter=20,  # start with 20 to get a feel for time
        scoring='neg_mean_squared_error',
        cv=3, #5, 
        random_state=random_state,
        n_jobs=-1,
        verbose=5  # 1
    )
    
    random_srch.fit(X_train, y_train)
    print('best xgb params:', random_srch.best_params_)
    # print('best score:', -random_srch.best_score_)
    best_xgb = random_srch.best_estimator_
    pickle.dump(best_xgb, open(xgb_rand_fp, 'wb'))
    
best_xgb = pickle.load(open(xgb_rand_fp,'rb'))
y_pred = best_xgb.predict(X_train)
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, y_pred)))
y_pred = best_xgb.predict(X_test)
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

#### Extract Feature Weights


In [ ]:

booster = best_xgb.get_booster()

importance_dict = booster.get_score(importance_type='gain')

imp_series = (
    pd.Series(importance_dict)
      .sort_values(ascending=False)
)

imp_arr = pd.Series(best_xgb.feature_importances_, index=X_train.columns)
top10 = imp_arr.sort_values(ascending=False).head(10)
print(top10)

# Conclusion